# Figure 2: perturbation-class coarse-state occupancy

This notebook creates the requested **perturbation-class occupancy bubble matrix** and the complete target-level supplementary heatmap from the processed CC-Perturb-seq H5AD on Sherlock.

It intentionally does **not** use target→program CRT effects, cNMF differential effects, or perturbation-driven clustering. State assignments are fixed before testing using the existing author-defined mapping of `leiden_res_0.5` recovered from `ipsc-ec.Hippo.figure.ipynb`:

| cluster | state |
|---:|---|
| 0 | artEC |
| 1 | iPSC |
| 2 | Neuro-iPSC |
| 3 | PrimStreak |
| 4 | Lat-Meso |
| 5 | Fibro-Mesen |
| 6 | Pre-EC |
| 7 | Neu-Mesen |
| 8 | Others |

### Primary model

For every day, target, and state, the notebook fits an adjusted one-vs-rest logistic regression:

```text
state membership ~ target-union carrier indicator
                 + biological sample
                 + pct_counts_mt
                 + log1p(total_counts)
                 + log1p(guides_per_cell)
                 + log1p(guide_umi_counts)
                 + log1p(n_genes_by_counts)
```

The main bubble matrix summarizes target-level models within reviewed regulator classes. The analysis never pools all guides in a class into one exposure.

## 0. Sherlock setup

Launch this notebook from the package root after obtaining an interactive allocation with enough memory and CPUs, for example:

```bash
srun --pty --time=24:00:00 --mem=128G --cpus-per-task=8 bash
source /path/to/your/environment/bin/activate
jupyter lab --no-browser --port=8888
```

The package also includes `scripts/run_full_occupancy_pipeline.sbatch` for a non-interactive run.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile
import yaml
import pandas as pd
from IPython.display import display, Image, Markdown

PACKAGE_DIR = Path.cwd().resolve()
if PACKAGE_DIR.name == "notebooks":
    PACKAGE_DIR = PACKAGE_DIR.parent

CONFIG_PATH = PACKAGE_DIR / "config" / "perturbation_class_occupancy_config.yaml"
EXPORT_DIR = PACKAGE_DIR / "occupancy_compact_export"
RESULT_DIR = PACKAGE_DIR / "occupancy_analysis_results"
N_JOBS = 8
FORCE_REEXPORT = False
FORCE_REFIT = False

print("PACKAGE_DIR:", PACKAGE_DIR)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXPORT_DIR:", EXPORT_DIR)
print("RESULT_DIR:", RESULT_DIR)


## 1. Configuration and preflight audit

The verified processed query object from the uploaded notebooks is:

```text
/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/2.preprocessing_tonys_pipeline/3.integrated_days/3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad
```

The analysis uses `intended_target_name` from the bundled guide metadata and a target-union carrier indicator.

In [ ]:
cfg = yaml.safe_load(CONFIG_PATH.read_text())
for key, value in cfg.items():
    print(f"{key}: {value}")

query_path = Path(cfg["query_h5ad"])
assert query_path.exists(), f"Query H5AD does not exist: {query_path}"

for rel_key in ["guide_metadata_tsv", "legacy_state_map_tsv", "class_membership_tsv", "state_definitions_tsv"]:
    p = Path(cfg[rel_key])
    if not p.is_absolute():
        p = PACKAGE_DIR / p
    assert p.exists(), f"Missing required package input: {p}"

print("\nAll configured inputs exist.")


In [ ]:
# Read only metadata/backed structure for a fast audit.
import anndata as ad
adata = ad.read_h5ad(query_path, backed="r")
print(adata)
print("\nobs columns:", adata.obs.columns.tolist())
print("\nobsm keys:", list(adata.obsm.keys()))
print("\nuns keys:", list(adata.uns.keys()))

required_obs = [
    cfg["cluster_col"], cfg["day_col"], cfg["replicate_col"],
    "pct_counts_mt", "total_counts", "guides_per_cell",
    "guide_umi_counts", "n_genes_by_counts",
]
missing_obs = [c for c in required_obs if c not in adata.obs.columns]
print("\nMissing required obs columns:", missing_obs)
assert not missing_obs
assert cfg["guide_assignment_key"] in adata.obsm
assert cfg["guide_names_key"] in adata.uns

display(adata.obs.groupby([cfg["day_col"], cfg["replicate_col"]], observed=True).size().rename("n_cells").reset_index())
display(adata.obs[cfg["cluster_col"]].astype(str).value_counts().sort_index().rename("n_cells").reset_index())

try:
    adata.file.close()
except Exception:
    pass


## 2. Review the fixed states and regulator classes

These tables are inputs, not results. The state boundaries are not re-estimated. Class membership is the literature-informed reviewed screen-hit annotation assembled earlier in this project; only targets represented in both the guide library and class table contribute to the class-level panel.

In [ ]:
def resolve_package_path(value):
    p = Path(value)
    return p if p.is_absolute() else PACKAGE_DIR / p

state_map = pd.read_csv(resolve_package_path(cfg["legacy_state_map_tsv"]), sep="\t", dtype={"source_cluster": str})
state_defs = pd.read_csv(resolve_package_path(cfg["state_definitions_tsv"]), sep="\t")
class_membership = pd.read_csv(resolve_package_path(cfg["class_membership_tsv"]), sep="\t")
guide_meta = pd.read_csv(resolve_package_path(cfg["guide_metadata_tsv"]), sep="\t")

print("Fixed cluster-to-state map")
display(state_map)
print("State definitions")
display(state_defs.sort_values("state_order"))
print("Reviewed regulator-class counts")
display(class_membership.groupby("regulator_class")["gene"].nunique().sort_values(ascending=False).rename("n_targets").reset_index())
print("Guide types")
display(guide_meta["type"].value_counts(dropna=False).rename("n_guides").reset_index())


## 3. Export compact cell metadata and target-union assignments

This step reads the H5AD once and exports only the metadata needed by the occupancy models.

Definitions encoded by the exporter:

- **Target carrier:** at least one assigned guide whose metadata maps to that `intended_target_name`.
- Multiple guides for the same target collapse to one target-union indicator; `n_guides_for_target` is retained.
- Cells with multiple biological targets contribute to each corresponding target indicator.
- **Pure control:** at least one safe/non-targeting guide and zero real-targeting guides.
- Cells carrying both control and real-targeting guides remain target carriers in the primary marginal model and are explicitly flagged.

In [ ]:
export_script = PACKAGE_DIR / "scripts" / "00_export_compact_occupancy_inputs.py"
if FORCE_REEXPORT or not (EXPORT_DIR / "export_audit.json").exists():
    cmd = [sys.executable, str(export_script), "--config", str(CONFIG_PATH)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=PACKAGE_DIR)
else:
    print("Compact export already exists; set FORCE_REEXPORT=True to rebuild it.")


In [ ]:
audit = json.loads((EXPORT_DIR / "export_audit.json").read_text())
print(json.dumps(audit, indent=2))

control_qc = pd.read_csv(EXPORT_DIR / "control_definition_qc.tsv", sep="\t")
state_counts = pd.read_csv(EXPORT_DIR / "state_counts_by_day_and_replicate.tsv", sep="\t")
carrier_counts = pd.read_csv(EXPORT_DIR / "target_carrier_counts_by_day.tsv", sep="\t")

print("Control-definition QC")
display(control_qc)
print("State counts by day and replicate")
display(state_counts.sort_values(["day", "biological_replicate", "assigned_state"]))
print("Carrier-count summary")
display(carrier_counts.groupby("day")["n_carrier_cells"].describe())

# Class coverage before modeling
exp_targets = set(carrier_counts["target_name"].astype(str))
class_cov = (
    class_membership.assign(in_experiment=class_membership["gene"].astype(str).isin(exp_targets))
    .groupby("regulator_class", as_index=False)
    .agg(n_reviewed_targets=("gene", "nunique"), n_targets_in_experiment=("in_experiment", "sum"))
    .sort_values("n_targets_in_experiment", ascending=False)
)
display(class_cov)


## 4. Fit per-target, per-day, per-state occupancy models

The primary population includes all cells from the relevant day. This is a marginal target-indicator model suited to the high-MOI design; it does not compare raw target fractions against a hand-selected control subset. Biological sample and the approved technical covariates are included in every model.

Benjamini–Hochberg adjustment is applied within each day across the target×state testing family. Replicate-specific models are fitted as a sensitivity check and combined by fixed-effect meta-analysis when estimable.

In [ ]:
fit_script = PACKAGE_DIR / "scripts" / "01_fit_occupancy_and_plot.py"
if FORCE_REFIT or not (RESULT_DIR / "target_state_occupancy_effects.tsv").exists():
    cmd = [
        sys.executable, str(fit_script),
        "--config", str(CONFIG_PATH),
        "--export-dir", str(EXPORT_DIR),
        "--output-dir", str(RESULT_DIR),
        "--n-jobs", str(N_JOBS),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=PACKAGE_DIR)
else:
    print("Model results already exist; set FORCE_REFIT=True to rerun all models.")


## 5. Audit target-level and class-level results

In [ ]:
effects = pd.read_csv(RESULT_DIR / "target_state_occupancy_effects.tsv", sep="\t")
rep_effects = pd.read_csv(RESULT_DIR / "target_state_occupancy_by_replicate.tsv", sep="\t")
class_summary = pd.read_csv(RESULT_DIR / "class_state_occupancy_summary.tsv", sep="\t")
coverage = pd.read_csv(RESULT_DIR / "coverage_and_filtering.tsv", sep="\t")

print("Target-level tests:", len(effects))
print("Converged:", int(effects["converged"].sum()), "/", len(effects))
print("FDR < 0.05:", int(effects["significant"].sum()))
print("Evaluable targets:", effects["target_name"].nunique())
print("Classes displayed:", class_summary["regulator_class"].nunique())

print("Coverage by day")
display(coverage.groupby("day").agg(
    n_targets=("target_name", "nunique"),
    n_passing=("passes_carrier_threshold", "sum"),
    median_carriers=("n_carrier_cells", "median"),
).reset_index())

print("Class-level summary")
display(class_summary.sort_values(["day", "state", "regulator_class"]))

if len(rep_effects):
    print("Replicate-specific models:", len(rep_effects))
    display(effects[[
        "day", "target_name", "state", "log_odds_ratio", "fdr",
        "replicate_sign_concordance", "n_replicates_evaluable"
    ]].sort_values("fdr").head(20))


## 6. Publication figures

In [ ]:
main_png = RESULT_DIR / "Fig2_perturbation_class_state_occupancy.png"
heatmap_png = RESULT_DIR / "Fig2_target_state_occupancy_heatmap_supplementary.png"

assert main_png.exists(), main_png
assert heatmap_png.exists(), heatmap_png

print("Main panel")
display(Image(filename=str(main_png)))
print("Supplementary target-level heatmap")
display(Image(filename=str(heatmap_png)))


## 7. Final output audit and packaging

The SVG exports retain text as text (`svg.fonttype = none`) and PDFs use TrueType fonts (`pdf.fonttype = 42`) for Illustrator compatibility.

In [ ]:
required_outputs = [
    "Fig2_perturbation_class_state_occupancy.svg",
    "Fig2_perturbation_class_state_occupancy.pdf",
    "Fig2_perturbation_class_state_occupancy.png",
    "Fig2_target_state_occupancy_heatmap_supplementary.svg",
    "Fig2_target_state_occupancy_heatmap_supplementary.pdf",
    "Fig2_target_state_occupancy_heatmap_supplementary.png",
    "target_state_occupancy_effects.tsv",
    "target_state_occupancy_by_replicate.tsv",
    "class_state_occupancy_summary.tsv",
    "class_membership.tsv",
    "state_definitions.tsv",
    "coverage_and_filtering.tsv",
    "draft_panel_legend.md",
    "methods_note.md",
]

missing = [name for name in required_outputs if not (RESULT_DIR / name).exists()]
print("Missing outputs:", missing)
assert not missing

# Check that SVG contains editable text nodes.
svg_text = (RESULT_DIR / "Fig2_perturbation_class_state_occupancy.svg").read_text(errors="ignore")
print("SVG contains <text> elements:", "<text" in svg_text)

zip_path = PACKAGE_DIR / "occupancy_analysis_results.zip"
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(RESULT_DIR))
print("Packaged results:", zip_path)


## Interpretation and reporting caveats

- These are occupancy effects in pre-existing coarse states, not target→program effects.
- State boundaries are the existing author-defined Leiden labels and are not rediscovered from carriers.
- The primary coefficient is the adjusted target-level log odds ratio from a day-specific one-vs-rest logistic model.
- Class dots summarize fitted target-level estimates; class guides are not pooled into a single exposure.
- The legacy state labels are hard assignments, so a genuine soft-probability sensitivity analysis is not available from this object.
- The main figure should be described as an **author-defined coarse-state occupancy analysis**, not as a new control-defined clustering analysis.